# Python编程基础 · 工单1.5《工业互联网数据采集系统》 — 教师参考版
第11周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/02-程序设计基础


## 参考实现 `w11`（含依赖导入）

In [2]:
import json
import random
import re
import sys
import numpy as np  # 通用科学计算


def w11():
    import time
    import json
    from pymodbus.client import ModbusTcpClient as Client
    import paho.mqtt.client as mqtt

    c = Client(host="127.0.0.1", port=5020)
    c.connect()
    mq = mqtt.Client()
    mq.connect("127.0.0.1", 1883)
    mq.loop_start()
    names = ["temp_c", "press_mpa", "vib_mm_s", "rpm", "status"]
    for i in range(5):
        rr = c.read_holding_registers(address=0, count=5)
        vals = rr.registers
        data = {n: (v / 10 if n != "status" else v)
                for n, v in zip(names, vals)}
        payload = json.dumps({"device_id": "DEV-1",
                              "ts": time.strftime("%H:%M:%S"),
                              "metrics": data}, ensure_ascii=False)
        mq.publish("device/1/status", payload)
        print("上报:", payload)
        time.sleep(2)
    mq.loop_stop()
    c.close()


## 学生版TODO → 教师答案对照
学生版starter中的TODO与上方参考实现（或下方双解）的对应关系：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `3` | TODO-3: 连接Broker(connect/loop_start), 失败时try捕获 |
| `5` | TODO-5(优秀): 订阅device/1/status回环验证(on_message打印回执) |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示（D层演示照做/C层只给思路行）。

## 多解对比（逻辑同构烟测，真连模拟器跑solutions.py w11）
| 解 | 路线 | 结论 |
|----|------|------|
| A | 同步轮询上报 | 简单，5轮×2s |
| B | 后台线程+MQTT回调 | 不阻塞主流程（payload同构已验） |

> 决策素材：低频巡检→A；产线实时→B。

## 运行环境(如需)

In [3]:
# 启动虚拟产线(离线模拟器, 5轮采集约15秒)
import subprocess, sys, time, atexit
p1 = subprocess.Popen([sys.executable, "虚拟产线/mini_modbus_server.py", "5020", "1"])
p2 = subprocess.Popen([sys.executable, "虚拟产线/mini_mqtt_broker.py", "1883"])
atexit.register(lambda: (p1.terminate(), p2.terminate()))
time.sleep(1.2)
print("模拟器就绪: PLC@5020 / MQTT@1883")

模拟器就绪: PLC@5020 / MQTT@1883


## 执行与验证

In [4]:
w11()

上报: {"device_id": "DEV-1", "ts": "06:58:53", "metrics": {"temp_c": 68.0, "press_mpa": 5.1, "vib_mm_s": 5.2, "rpm": 14.8, "status": 1}}


/var/folders/6j/khvhkzp55m30fx539sm9tggr0000gn/T/ipykernel_68516/2315781391.py:16: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  mq = mqtt.Client()


上报: {"device_id": "DEV-1", "ts": "06:58:55", "metrics": {"temp_c": 68.7, "press_mpa": 5.4, "vib_mm_s": 5.4, "rpm": 14.8, "status": 1}}


上报: {"device_id": "DEV-1", "ts": "06:58:57", "metrics": {"temp_c": 69.2, "press_mpa": 5.6, "vib_mm_s": 5.4, "rpm": 14.8, "status": 1}}


上报: {"device_id": "DEV-1", "ts": "06:58:59", "metrics": {"temp_c": 69.6, "press_mpa": 5.9, "vib_mm_s": 5.3, "rpm": 14.8, "status": 1}}


上报: {"device_id": "DEV-1", "ts": "06:59:01", "metrics": {"temp_c": 69.9, "press_mpa": 6.1, "vib_mm_s": 5.0, "rpm": 14.9, "status": 1}}
